In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.9 MB/s eta 0:00:00


In [ ]:
!pip install -q yt-dlp pydub pandas tqdm mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 21.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
from google.colab import drive
from mistralai import Mistral
from tqdm.notebook import tqdm
from google.colab import userdata

print("Montando Google Drive per salvare i dati in modo sicuro...")
drive.mount('/content/drive')



print("✅ Setup completato!")

Montando Google Drive per salvare i dati in modo sicuro...
Mounted at /content/drive
✅ Setup completato!


In [ ]:
MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
DATA_PATH = "/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v4/bologna_training_data_v4.jsonl"

In [ ]:
DATA_PATH

'/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v4/bologna_training_data_v4.jsonl'

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
print("✅ Logged into Hugging Face!")

✅ Logged into Hugging Face!


In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Mistral3ForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import torch

MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-v4-50"

# ── 1. Tokenizer ──────────────────────────────────────────────────────────────
print("1. Caricamento Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── 2. Model in 4-bit, tutto fp16 ────────────────────────────────────────────
print("2. Caricamento Modello in 4-bit fp16...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # <-- fp16, not bf16
)

model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,              # <-- fp16, not bf16
)
model.config.use_cache = False              # required for gradient checkpointing
model = prepare_model_for_kbit_training(model)

# ── 3. LoRA adapter ───────────────────────────────────────────────────────────
print("3. Applicazione Adapter LoRA...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ── 3.5. Purging any surviving bf16 tensors (params + buffers only) ─────────────
print("3.5. Purga bf16 residui...")
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

for name, buf in model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

# ── 4. Dataset ────────────────────────────────────────────────────────────────
print("4. Preparazione Dataset...")
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"✅ {len(dataset)} esempi trovati.")

def format_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example

formatted_dataset = dataset.map(format_chat_template)
split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

print("5. Configurazione Trainer...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_strategy="steps",
    logging_steps=10,
    num_train_epochs=2,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    save_strategy="steps",
    save_steps=10,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,
    eval_strategy="steps",
    eval_steps=10,
    load_best_model_at_end=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)



1. Caricamento Tokenizer...
2. Caricamento Modello in 4-bit fp16...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

3. Applicazione Adapter LoRA...
trainable params: 12,517,376 || all params: 3,861,607,424 || trainable%: 0.3241
3.5. Purga bf16 residui...
4. Preparazione Dataset...
✅ 1548 esempi trovati.


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train: 1393, Eval: 155
5. Configurazione Trainer...


In [ ]:
OUTPUT_DIR

'/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-v4-50'

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
print("🚀 Partenza Addestramento!")
trainer.train()

print("✅ Salvataggio adapter LoRA...")
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("✅ Fatto!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 2}.


🚀 Partenza Addestramento!


Step,Training Loss,Validation Loss
10,3.132545,2.717367
20,2.088793,1.016696
30,0.396779,0.055811


KeyboardInterrupt: 

In [ ]:
OUTPUT_DIR

'/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-v4-30'

In [ ]:
print("✅ Salvataggio adapter LoRA...")
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("✅ Fatto!")

✅ Salvataggio adapter LoRA...
✅ Fatto!


In [ ]:
# 1. Montiamo Google Drive per accedere al modello salvato
from google.colab import drive
drive.mount('/content/drive')

# 2. Facciamo il login su Hugging Face usando il tuo token segreto
from huggingface_hub import login, HfApi
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# 3. Definiamo la cartella dove si trova l'adapter
OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-v4-50"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# 4. Inizializziamo l'API e carichiamo!
api = HfApi()

REPO_NAME = "n1kg0r/Vezstral-3B-Bolognese-v4-50"

print(f"Creazione della repository {REPO_NAME} su Hugging Face...")
api.create_repo(repo_id=REPO_NAME, exist_ok=True)

print("Caricamento dell'adapter in corso...")
api.upload_folder(
    folder_path=f"{OUTPUT_DIR}/final_adapter",
    repo_id=REPO_NAME,
    repo_type="model",
)
print(f"✅ Fatto! Vezstral è live su: https://huggingface.co/{REPO_NAME}")

Creazione della repository n1kg0r/Vezstral-3B-Bolognese-v4-30 su Hugging Face...
Caricamento dell'adapter in corso...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al_adapter/tokenizer.json:  94%|#########3| 16.0MB / 17.1MB            

  ...adapter_model.safetensors:   3%|2         |  703kB / 25.1MB            

✅ Fatto! Vezstral è live su: https://huggingface.co/n1kg0r/Vezstral-3B-Bolognese-v4-30
